In [ ]:
from fedbiomed.common.training_plans import FedSGDClassifier
import pandas as pd
from fedbiomed.common.dataset import NipoppyDataset

class SGDClassifierTrainingPlan(FedSGDClassifier):
    TERMURL_AGE = "nb:Age"
    TERMURL_SEX = "nb:Sex"
    TERMURL_COG_DECLINE = "fl:cognitive_decline_status"
    TERMURL_COG_DECLINE_AVAILABILITY = "fl:cognitive_decline_availability"
    TERMURL_DIAGNOSIS = "nb:Diagnosis"

    # values
    TERMURL_AVAILABLE = "nb:available"
    TERMURL_UNAVAILABLE = "nb:unavailable"
    TERMURL_MALE = "snomed:248153007"
    TERMURL_FEMALE = "snomed:248152002"
    TERMURL_HEALTHY_CONTROL = "ncit:C94342"

    def init_dependencies(self):
        deps = [
                "from skrub import TableVectorizer",
                "from sklearn.preprocessing import OneHotEncoder",
                "import pandas as pd",
                "from fedbiomed.common.dataset import NipoppyDataset",
               ]
        return deps

    @staticmethod
    def transform_skrub(df: pd.DataFrame) -> pd.DataFrame:
        specific_transformers = []
        if SGDClassifierTrainingPlan.TERMURL_SEX in df.columns:
            specific_transformers.append(
                (
                    OneHotEncoder(
                        drop=[SGDClassifierTrainingPlan.TERMURL_MALE], sparse_output=False
                    ),
                    [SGDClassifierTrainingPlan.TERMURL_SEX],
                )
            )
        if SGDClassifierTrainingPlan.TERMURL_COG_DECLINE in df.columns:
            specific_transformers.append(
                (
                    OneHotEncoder(
                        drop=[SGDClassifierTrainingPlan.TERMURL_UNAVAILABLE],
                        sparse_output=False,
                        feature_name_combiner=lambda x, _: x,
                    ),
                    [SGDClassifierTrainingPlan.TERMURL_COG_DECLINE],
                )
            )

        table_vectorizer = TableVectorizer(specific_transformers=specific_transformers)
        df = table_vectorizer.fit_transform(df)
        return df
    
    def training_data(self):
        FS_NAME = "freesurfer"
        FS_STATS_NAME = "fs_stats"
        SUFFIX_APARC = "-aparc.DKTatlas-thickness.tsv"
        SUFFIX_ASEG = "-aseg-volume.tsv"
        sess_filter = '01' if self.node_id() == 'NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20' else 'baseline'
        dataset = NipoppyDataset(phenotypes=[
                                    SGDClassifierTrainingPlan.TERMURL_AGE,
                                    SGDClassifierTrainingPlan.TERMURL_SEX,
                                    SGDClassifierTrainingPlan.TERMURL_DIAGNOSIS,
                                    SGDClassifierTrainingPlan.TERMURL_COG_DECLINE_AVAILABILITY,
                                ],
                                derivatives=[("freesurfer",
                                              "7.3.2",
                                              "idp/fs_stats-0.2.1/fs7.3.2-aparc.DKTatlas-thickness.tsv",)],
                                target=[SGDClassifierTrainingPlan.TERMURL_COG_DECLINE],
                                session_filters=sess_filter,  # session filter
                                drop_na=True,
                                sample_level_transform=None,
                                sample_level_target_transform=None,
                                whole_df_level_transform=SGDClassifierTrainingPlan.transform_skrub)
        FS_VERSION = dataset.
        FS_STATS_VERSION = "0.2.1"
        dataset.derivatives = [(FS_NAME, FS_VERSION, f"idp/fs_stats-{FS_STATS_VERSION}/fs{FS_VERSION}-{SUFFIX_APARC}")]

        return DataManager(dataset=dataset, shuffle=True)


In [2]:
# harcoded for now... how to get a better idea in the future?
n_features = 71

In [3]:
from fedbiomed.researcher.requests import Requests
from fedbiomed.researcher.config import config

all_datasets = Requests(config).list()


2026-05-20 12:47:47,601 fedbiomed INFO - Syslog configuration is disabled or not found in configuration. Disabling syslog logging.

2026-05-20 12:47:47,612 fedbiomed INFO - Starting researcher service...

2026-05-20 12:47:47,612 fedbiomed INFO - Waiting 3s for nodes to connect...

In [4]:
all_datasets

{'NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b22': [{'name': 'nipoppy',
   'data_type': 'nipoppy',
   'tags': ['nipoppy'],
   'description': 'ni',
   'dataset_id': 'dataset_9f858453-9ab1-49e2-9233-1c09f140fcd6',
   'dataset_parameters': {}},
  {'name': 'h',
   'data_type': 'nipoppy',
   'tags': ['h'],
   'description': 'j',
   'shape': {'nipoppy': [1, 1]},
   'dataset_id': 'dataset_8e4af98e-cb63-4f36-84e9-497ae699f037',
   'dataset_parameters': {}},
  {'name': 's',
   'data_type': 'nipoppy',
   'tags': ['s'],
   'description': 'a',
   'shape': {'nipoppy': [299, 6]},
   'dataset_id': 'dataset_0d83ad5c-b6a0-42e6-a896-212c00766223',
   'dataset_parameters': {}},
  {'name': 'ss',
   'data_type': 'nipoppy',
   'tags': ['nipoppy2'],
   'description': '',
   'shape': {'nipoppy': [299, 1]},
   'dataset_id': 'dataset_c4efc430-f843-435a-9f57-04d6105e9d38',
   'dataset_parameters': {}},
  {'name': 'ni',
   'data_type': 'nipoppy',
   'tags': ['nipoppy3'],
   'description': '',
   'shape': {'nipoppy': [

In [5]:
model_args = {
            # fedbiomed
            "eta0": 0.05,
            "random_state": 424242,
            "n_features": n_features,
            "n_classes": 2,
            # model
            "learning_rate": "invscaling",
            "penalty": "l2",
        }

In [6]:
training_args = {
    'num_updates': 5,  # 50,
    "loader_args": {"batch_size": 50}
}

In [7]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage

tags =  ['nipoppy3']
rounds = 2

# search for corresponding datasets across nodes datasets
exp = Experiment(tags=tags,
                 model_args=model_args,
                 training_plan_class=SGDClassifierTrainingPlan,
                 training_args=training_args,
                 round_limit=rounds,
                 aggregator=FedAverage(),
                 node_selection_strategy=None)

2026-05-20 12:47:51,082 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-05-20 12:47:51,091 fedbiomed INFO - Node selected for training -> Default Node Name 2
Node ID is -> NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b22

2026-05-20 12:47:51,092 fedbiomed INFO - Node selected for training -> Default Node Name
Node ID is -> NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20

I0000 00:00:1779274071.762628  389997 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


2026-05-20 12:47:51,914 fedbiomed WARNING - Option share_persistent_buffers is not supported in SKLearnTrainingPlan, it will be ignored.

In [ ]:
exp.run()